# NDR - Notebook 34: 229 base + MULTI-SCALE recall recovery (tan cong FN)

## Can cu (research + thuc nghiem)
- Paper ESA (chinh chu, cung BTC): poison thiet ke de f_poison(X) ~ f_clean(X) cho input sach.
  => discriminator bi chan boi THIET KE (giai thich moi filter cap ~229). Khong danh vao day nua.
- Toan metric: empty=284 = TOAN BO khoi luong streak that. 229->145 can cat ~84 diem,
  phan lon la FN cua streak that (streak clean model thay ma poisoned model BO SOT @thr0.05).
- LEVER CHUA AI CHAM: moi NB (cua ta + 7 doi thu) chi lay candidate tu poisoned model @0.05 (1 scale)
  va CHI XOA. Chua ai THEM candidate. Baseline chay o MIN_SIZE_TEST=800 (default d2) < anh 1024.
  Chay da ti le (800/1024/1333) + WBF fuse -> thu hoi streak mo/nho bi miss -> cat FN.

## Guardrails (KHONG lap sai lam cu)
- KHONG chon ban nop theo poison_mass (NB33b: guard giam nhung LB TE - 240). Chi in tham khao.
- KHONG gate bang val_macadd local (pseudo-clean anti-correlated - lua 4 lan).
- GIU rescore 229 NGUYEN (s_diff 0.9 + s_geo 0.1, demote p>=0.55). Chi doi 1 bien: NGUON candidate.
- Tieu chi: density sau rescore + so candidate THEM song qua rescore. So LB voi 229 anchor.

## Xuat DUNG 1 file submission.csv (+ anchor 229 single-scale de A/B)


## 1. Setup

In [1]:
!pip install -q 'setuptools<81'
!pip install -q 'git+https://github.com/facebookresearch/detectron2.git'
!pip install -q ensemble_boxes


  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 1.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.5/155.5 kB 5.3 MB/s eta 0:00:00


In [2]:
import copy, json, os, random
from pathlib import Path
import cv2, numpy as np, pandas as pd, torch, torch.nn as nn
from scipy.optimize import linear_sum_assignment
from tqdm import tqdm
from detectron2 import model_zoo
from detectron2.config import get_cfg
from detectron2.data import (DatasetCatalog, DatasetMapper, MetadataCatalog,
                             build_detection_train_loader, detection_utils as utils)
from detectron2.engine import DefaultPredictor, DefaultTrainer
from detectron2.modeling import build_model
from detectron2.checkpoint import DetectionCheckpointer
import logging; logging.getLogger('detectron2').setLevel(logging.ERROR)

ROOT = '/kaggle/input/competitions/neural-debris-removal-in-streak-detection-models'
POISONED_WEIGHTS = f'{ROOT}/poisoned_model/poisoned_model.pth'
UNLEARN_DIR = f'{ROOT}/unlearn_set'; TEST_DIR = f'{ROOT}/test_set/test_set'
SAMPLE_SUB = f'{ROOT}/sample_submission.csv'
WORK = Path('/kaggle/working'); WORK.mkdir(exist_ok=True)

BASE_CONFIG = 'COCO-Detection/retinanet_R_50_FPN_3x.yaml'
ANCHOR_ASPECT_RATIOS = [0.1, 0.2, 0.5, 1.0, 2.0, 5.0, 10.0]
ANCHOR_SIZES = [[16], [32], [64], [128], [256]]
NUM_CLASSES = 1; IMG_W = IMG_H = 1024; BATCH_SIZE = 4

# --- 229 exact knobs ---
PRUNE_FRAC = 0.15
UNLEARN_LR = 2.5e-4
UNLEARN_ITERS = 20
EWC_LAMBDA_SWEEP = [500.0, 1000.0, 2000.0, 4000.0]   # 500 = 229 goc

A_FACTOR = 10.0
IOU_THRESHOLDS = np.arange(0.2, 0.91, 0.1)
CAND_THRESH = 0.05
# --- 229 rescore knobs ---
R = {'MIN_KEEP':0.20, 'P_HI':0.55, 'P_LO':0.25, 'EPS':0.01}
W_DIFF, W_GEO = 0.90, 0.10

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Setup OK | device:', DEVICE, '| EWC sweep:', EWC_LAMBDA_SWEEP)


Setup OK | device: cuda | EWC sweep: [500.0, 1000.0, 2000.0, 4000.0]


## 2. Data + helpers (229 exact)

In [3]:
def load_image(path):
    im = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)
    if im.dtype == np.uint16: im = im.astype(np.float32)/65535.0
    im = np.clip(im*255,0,255).astype(np.float32)
    if im.ndim==2: im = np.repeat(im[:,:,None],3,axis=2)
    return im

def build_cfg(weights=POISONED_WEIGHTS, score_thresh=CAND_THRESH):
    cfg = get_cfg(); cfg.merge_from_file(model_zoo.get_config_file(BASE_CONFIG))
    cfg.MODEL.WEIGHTS = weights; cfg.MODEL.RETINANET.NUM_CLASSES = NUM_CLASSES
    cfg.MODEL.RETINANET.SCORE_THRESH_TEST = score_thresh
    cfg.MODEL.ANCHOR_GENERATOR.ASPECT_RATIOS = [ANCHOR_ASPECT_RATIOS]
    cfg.MODEL.ANCHOR_GENERATOR.SIZES = ANCHOR_SIZES
    cfg.MODEL.DEVICE = DEVICE
    return cfg
def build_predictor(weights, score_thresh=CAND_THRESH): return DefaultPredictor(build_cfg(weights, score_thresh))

UNLEARN_DATASET = 'unlearn'
def register_unlearn(unlearn_dir):
    with open(Path(unlearn_dir)/'annotations_coco.json') as f: coco = json.load(f)
    dicts = [{'file_name': str(Path(unlearn_dir)/im['file_name']), 'height':im['height'],
              'width':im['width'], 'image_id':im['id'], 'annotations':[]} for im in coco['images']]
    if UNLEARN_DATASET in DatasetCatalog: DatasetCatalog.remove(UNLEARN_DATASET)
    DatasetCatalog.register(UNLEARN_DATASET, lambda: dicts)
    MetadataCatalog.get(UNLEARN_DATASET).set(thing_classes=['object'])
    return dicts
unlearn_dicts = register_unlearn(UNLEARN_DIR)
with open(Path(UNLEARN_DIR)/'annotations_coco.json') as f: coco_data = json.load(f)
poison_boxes = {}
for ann in coco_data['annotations']:
    poison_boxes.setdefault(ann['image_id'], []).append(ann['bbox'])

class UInt16DatasetMapper(DatasetMapper):
    def __call__(self, dd):
        dd = copy.deepcopy(dd); image = load_image(dd['file_name'])
        dd['image'] = torch.as_tensor(image.transpose(2,0,1).copy())
        dd['instances'] = utils.annotations_to_instances([], image.shape[:2])
        return dd

unlearn_files = sorted(Path(UNLEARN_DIR).glob('*.png'))
test_files = sorted(Path(TEST_DIR).glob('*.png'))
print('unlearn:', len(unlearn_files), '| test:', len(test_files))


unlearn: 20 | test: 2000


## 3. maCADD metric + iou

In [4]:
def iou_matrix(a, b):
    if len(a)==0 or len(b)==0: return np.zeros((len(a),len(b)),np.float32)
    ax1,ay1,ax2,ay2 = a[:,0:1],a[:,1:2],a[:,2:3],a[:,3:4]
    bx1,by1,bx2,by2 = b[:,0],b[:,1],b[:,2],b[:,3]
    iw = np.maximum(0,np.minimum(ax2,bx2)-np.maximum(ax1,bx1))
    ih = np.maximum(0,np.minimum(ay2,by2)-np.maximum(ay1,by1))
    inter = iw*ih
    aa = np.maximum(0,ax2-ax1)*np.maximum(0,ay2-ay1); ab = np.maximum(0,bx2-bx1)*np.maximum(0,by2-by1)
    union = aa+ab-inter
    return np.where(union>0, inter/union, 0.0).astype(np.float32)

def acadd_t(cb, cs, pb, ps, t, cip=None):
    if cip is None: cip = np.zeros(len(cb), bool)
    ious = iou_matrix(cb, pb); cost = 0.0; mc,mp = set(),set()
    if ious.size:
        ri,ci = linear_sum_assignment(-ious)
        for i,j in zip(ri,ci):
            if ious[i,j] >= t:
                mc.add(i); mp.add(j); d = cs[i]-ps[j]
                if cip[i]: cost += (d/A_FACTOR) if d>0 else -d
                else:      cost += d if d>0 else (-d/A_FACTOR)
    cost += sum(cs[i] for i in range(len(cs)) if i not in mc)
    cost += sum(ps[j] for j in range(len(ps)) if j not in mp)
    return cost

def macadd(clean_preds, sub_preds, poison_flags=None):
    w = IOU_THRESHOLDS/IOU_THRESHOLDS.sum(); per=[]
    for stem,(cb,cs) in clean_preds.items():
        keep = cs>0.2; cb,cs = cb[keep],cs[keep]
        pf = poison_flags.get(stem)[keep] if poison_flags else None
        pb,ps = sub_preds.get(stem,(np.zeros((0,4)),np.zeros(0)))
        per.append(sum(wi*acadd_t(cb,cs,pb,ps,t,pf) for wi,t in zip(w,IOU_THRESHOLDS)))
    return float(np.mean(per))
print('metric ready')


metric ready


## 4. Activation-guided pruning (1 LAN)

In [5]:
def collect_activations(model, unlearn_dicts, poison_boxes):
    model.eval()
    target = [m for m in model.head.cls_subnet if isinstance(m, nn.Conv2d)]
    hooks, stored = [], {}
    for i, layer in enumerate(target):
        stored[i] = []
        hooks.append(layer.register_forward_hook(lambda m,inp,out,idx=i: stored[idx].append(out.detach().cpu())))
    with torch.no_grad():
        for d in tqdm(unlearn_dicts, desc='acts'):
            im = load_image(d['file_name'])
            model([{'image': torch.as_tensor(im.transpose(2,0,1)).to(DEVICE)}])
    for h in hooks: h.remove()
    return stored

def compute_poison_scores(stored, unlearn_dicts, poison_boxes):
    spl = {}
    for li, acts in stored.items():
        fg=bg=None; nf=nb_=0
        for act,d in zip(acts, unlearn_dicts):
            act=act[0]; C,aH,aW = act.shape
            sy,sx = aH/d['height'], aW/d['width']; boxes = poison_boxes.get(d['image_id'],[])
            if fg is None: fg=torch.zeros(C); bg=torch.zeros(C)
            for x,y,w,h in boxes:
                x1=max(0,int(x*sx)); y1=max(0,int(y*sy)); x2=min(aW,int((x+w)*sx)+1); y2=min(aH,int((y+h)*sy)+1)
                if x2>x1 and y2>y1: fg += act[:,y1:y2,x1:x2].relu().mean([1,2]); nf+=1
            rng = np.random.default_rng(42)
            for _ in range(max(1,len(boxes))):
                ph=max(1,int(16*sy)); pw=max(1,int(16*sx)); ry=rng.integers(0,max(1,aH-ph)); rx=rng.integers(0,max(1,aW-pw))
                bg += act[:,ry:ry+ph,rx:rx+pw].relu().mean([1,2]); nb_+=1
        if nf: fg/=nf
        if nb_: bg/=nb_
        spl[li] = (fg-bg).numpy()
    return spl

def prune_channels(model, spl, prune_frac=PRUNE_FRAC):
    target = [(i,m) for i,m in enumerate(model.head.cls_subnet) if isinstance(m, nn.Conv2d)]
    tot=0
    for i,layer in target:
        if i not in spl: continue
        sc = spl[i]; n_pr = max(1,int(len(sc)*prune_frac)); bad = np.argsort(sc)[-n_pr:]
        with torch.no_grad():
            layer.weight.data[bad]=0.0
            if layer.bias is not None: layer.bias.data[bad]=0.0
        tot+=n_pr
    print(f'pruned {tot} channels'); return model

cfg = build_cfg(); model = build_model(cfg); DetectionCheckpointer(model).load(POISONED_WEIGHTS); model=model.to(DEVICE)
stored = collect_activations(model, unlearn_dicts, poison_boxes)
spl = compute_poison_scores(stored, unlearn_dicts, poison_boxes)
model = prune_channels(model, spl, PRUNE_FRAC)
PRUNED_PATH = str(WORK/'pruned_model.pth')
torch.save({'model': model.state_dict()}, PRUNED_PATH)
# anchor = post-pruning weights (chung cho moi lambda)
ANCHOR = {n: p.detach().clone() for n,p in model.named_parameters() if ('cls_score' in n or 'cls_subnet' in n)}
del model; torch.cuda.empty_cache()
print('pruned model saved ->', PRUNED_PATH)


Loading config /usr/local/lib/python3.12/dist-packages/detectron2/model_zoo/configs/COCO-Detection/../Base-RetinaNet.yaml with yaml.unsafe_load. Your machine may be at risk if the file contains malicious content.
acts:   0%|          | 0/20 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/functional.py:505: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4381.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
acts: 100%|██████████| 20/20 [00:05<00:00,  3.63it/s]


pruned 76 channels
pruned model saved -> /kaggle/working/pruned_model.pth


## 5. Train 1 de-poison model (229 exact: lr2.5e-4, 20it, cls-only EWC lambda500)

In [6]:
class ClsEWCTrainer(DefaultTrainer):
    anchor_weights=None; ewc_lambda=500.0
    @classmethod
    def build_train_loader(cls, cfg):
        return build_detection_train_loader(cfg, mapper=UInt16DatasetMapper(cfg, is_train=True, augmentations=[]),
                                            dataset=DatasetCatalog.get(cfg.DATASETS.TRAIN[0]))
    @classmethod
    def build_model(cls, cfg):
        m=super().build_model(cfg)
        for n,p in m.named_parameters(): p.requires_grad=("cls_score" in n or "cls_subnet" in n)
        return m
    def run_step(self):
        assert self.model.training
        if not hasattr(self,"_dli"): self._dli=iter(self.data_loader)
        try: data=next(self._dli)
        except StopIteration: self._dli=iter(self.data_loader); data=next(self._dli)
        ld=self.model(data)
        if self.anchor_weights is not None:
            ewc=torch.tensor(0.0,device=DEVICE)
            for n,p in self.model.named_parameters():
                if p.requires_grad and n in self.anchor_weights: ewc=ewc+((p-self.anchor_weights[n])**2).sum()
            ld["loss_ewc"]=self.ewc_lambda*ewc
        loss=sum(ld.values()); self.optimizer.zero_grad(); loss.backward(); self.optimizer.step()

out=str(WORK/"dep229"); Path(out).mkdir(parents=True, exist_ok=True)
cfg=build_cfg(weights=PRUNED_PATH); cfg.DATASETS.TRAIN=(UNLEARN_DATASET,); cfg.DATASETS.TEST=()
cfg.DATALOADER.NUM_WORKERS=2; cfg.SOLVER.IMS_PER_BATCH=BATCH_SIZE
cfg.SOLVER.BASE_LR=UNLEARN_LR; cfg.SOLVER.MAX_ITER=UNLEARN_ITERS; cfg.SOLVER.STEPS=[]; cfg.OUTPUT_DIR=out
ClsEWCTrainer.anchor_weights=ANCHOR; ClsEWCTrainer.ewc_lambda=500.0
tr=ClsEWCTrainer(cfg); tr.resume_or_load(resume=False); tr.train()
DEP_PATH=str(Path(out)/"model_final.pth"); print("depoison model ->", DEP_PATH)

Loading config /usr/local/lib/python3.12/dist-packages/detectron2/model_zoo/configs/COCO-Detection/../Base-RetinaNet.yaml with yaml.unsafe_load. Your machine may be at risk if the file contains malicious content.


[07/16 13:07:11 d2.engine.defaults]: Model:
RetinaNet(
  (backbone): FPN(
    (fpn_lateral3): Conv2d(512, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output3): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral4): Conv2d(1024, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output4): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (fpn_lateral5): Conv2d(2048, 256, kernel_size=(1, 1), stride=(1, 1))
    (fpn_output5): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (top_block): LastLevelP6P7(
      (p6): Conv2d(2048, 256, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
      (p7): Conv2d(256, 256, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1))
    )
    (bottom_up): ResNet(
      (stem): BasicStem(
        (conv1): Conv2d(
          3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False
          (norm): FrozenBatchNorm2d(num_features=64, eps=1e-05)
        )
      )
      (res2)

2026-07-16 13:07:25.545753: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1784207245.742439      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1784207245.803167      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1784207246.296715      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1784207246.296745      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1784207246.296748      24 computation_placer.cc:177] computation placer alr

[07/16 13:07:38 d2.engine.hooks]: Overall training speed: 18 iterations in 0:00:09 (0.5024 s / it)
[07/16 13:07:38 d2.engine.hooks]: Total training time: 0:00:24 (0:00:15 on hooks)
depoison model -> /kaggle/working/dep229/model_final.pth


## 6. Cache candidates + depoison preds + signals (229 exact)

In [7]:
poisoned_pred=build_predictor(POISONED_WEIGHTS, CAND_THRESH)
dep_pred=build_predictor(DEP_PATH, CAND_THRESH)
def pr(predictor, img):
    inst=predictor(img)["instances"].to("cpu")
    return inst.pred_boxes.tensor.numpy().astype(np.float32), inst.scores.numpy().astype(np.float32)

TEST_CAND={p.stem: pr(poisoned_pred, load_image(p)) for p in tqdm(test_files, desc="cand test")}
TEST_DEP ={p.stem: pr(dep_pred, load_image(p)) for p in tqdm(test_files, desc="dep test")}

logwh=np.log(np.array([ann["bbox"][2:4] for ann in coco_data["annotations"]], np.float32))
MU=logwh.mean(0); COV=np.cov(logwh.T)+1e-3*np.eye(2); COV_INV=np.linalg.inv(COV)
def geometry_scores(boxes):
    if len(boxes)==0: return np.zeros(0,np.float32)
    wh=np.stack([boxes[:,2]-boxes[:,0], boxes[:,3]-boxes[:,1]],1)
    d=np.log(np.maximum(wh,1e-3))-MU; m2=np.einsum("ni,ij,nj->n", d, COV_INV, d)
    return np.exp(-0.5*m2/2.0).astype(np.float32)
def s_diff_single(cb,cs,db,ds):
    sd=np.ones(len(cb),np.float32)
    if len(cb) and len(db):
        ious=iou_matrix(cb,db); j=ious.argmax(1); bi=ious[np.arange(len(cb)),j]
        for i in range(len(cb)):
            if bi[i]>=0.5: sd[i]=1.0-(ds[j[i]]/max(cs[i],1e-6))
    return np.clip(sd,0,1)
print("signals ready")

Loading config /usr/local/lib/python3.12/dist-packages/detectron2/model_zoo/configs/COCO-Detection/../Base-RetinaNet.yaml with yaml.unsafe_load. Your machine may be at risk if the file contains malicious content.


[07/16 13:07:38 d2.checkpoint.detection_checkpoint]: [DetectionCheckpointer] Loading from /kaggle/input/competitions/neural-debris-removal-in-streak-detection-models/poisoned_model/poisoned_model.pth ...


Loading config /usr/local/lib/python3.12/dist-packages/detectron2/model_zoo/configs/COCO-Detection/../Base-RetinaNet.yaml with yaml.unsafe_load. Your machine may be at risk if the file contains malicious content.


[07/16 13:07:39 d2.checkpoint.detection_checkpoint]: [DetectionCheckpointer] Loading from /kaggle/working/dep229/model_final.pth ...


dep test: 100%|██████████| 2000/2000 [03:33<00:00,  9.35it/s]

signals ready


## 7. Multi-scale + WBF predict (recall recovery). Nguon candidate MOI.

In [8]:
from ensemble_boxes import weighted_boxes_fusion

# scales: (min_size, max_size). 800=default d2 (=229 baseline). 1024/1333 = res cao hon -> bat streak mo/nho.
SCALES = [(800,1333),(1024,1707),(1333,2222)]

def build_predictor_scale(weights, min_size, max_size, score_thresh=CAND_THRESH):
    cfg = build_cfg(weights, score_thresh)
    cfg.INPUT.MIN_SIZE_TEST = min_size; cfg.INPUT.MAX_SIZE_TEST = max_size
    return DefaultPredictor(cfg)

def predict_ms(predictors, img, iou_thr=0.55):
    """WBF-fuse detections tu nhieu scale. Tra (boxes_xyxy, scores)."""
    H,W = img.shape[:2]; bl,sl,ll=[],[],[]
    for prd in predictors:
        b,s = pr(prd, img)
        if len(b)==0: continue
        bn=b.copy(); bn[:,[0,2]]=np.clip(bn[:,[0,2]],0,W)/W; bn[:,[1,3]]=np.clip(bn[:,[1,3]],0,H)/H
        bl.append(bn.tolist()); sl.append(s.tolist()); ll.append([0]*len(s))
    if not bl: return np.zeros((0,4),np.float32), np.zeros(0,np.float32)
    fb,fs,_ = weighted_boxes_fusion(bl,sl,ll, iou_thr=iou_thr, skip_box_thr=CAND_THRESH)
    fb=fb.copy(); fb[:,[0,2]]*=W; fb[:,[1,3]]*=H
    return fb.astype(np.float32), fs.astype(np.float32)

# cache multi-scale candidate (poisoned) + dep (cho s_diff) tren test
pois_ms = [build_predictor_scale(POISONED_WEIGHTS, mn, mx) for mn,mx in SCALES]
TEST_CAND_MS = {p.stem: predict_ms(pois_ms, load_image(p)) for p in tqdm(test_files, desc="cand MS test")}
del pois_ms; torch.cuda.empty_cache()

dep_ms = [build_predictor_scale(DEP_PATH, mn, mx) for mn,mx in SCALES]
TEST_DEP_MS = {p.stem: predict_ms(dep_ms, load_image(p)) for p in tqdm(test_files, desc="dep MS test")}
del dep_ms; torch.cuda.empty_cache()

n_single = sum(len(s) for _,s in TEST_CAND.values())
n_ms      = sum(len(s) for _,s in TEST_CAND_MS.values())
print(f"candidate pool @0.05: single-scale={n_single} ({n_single/len(test_files):.3f}/img) | "
      f"multi-scale={n_ms} ({n_ms/len(test_files):.3f}/img) | THEM {n_ms-n_single} box")


Loading config /usr/local/lib/python3.12/dist-packages/detectron2/model_zoo/configs/COCO-Detection/../Base-RetinaNet.yaml with yaml.unsafe_load. Your machine may be at risk if the file contains malicious content.


[07/16 13:15:17 d2.checkpoint.detection_checkpoint]: [DetectionCheckpointer] Loading from /kaggle/input/competitions/neural-debris-removal-in-streak-detection-models/poisoned_model/poisoned_model.pth ...


Loading config /usr/local/lib/python3.12/dist-packages/detectron2/model_zoo/configs/COCO-Detection/../Base-RetinaNet.yaml with yaml.unsafe_load. Your machine may be at risk if the file contains malicious content.


[07/16 13:15:17 d2.checkpoint.detection_checkpoint]: [DetectionCheckpointer] Loading from /kaggle/input/competitions/neural-debris-removal-in-streak-detection-models/poisoned_model/poisoned_model.pth ...


Loading config /usr/local/lib/python3.12/dist-packages/detectron2/model_zoo/configs/COCO-Detection/../Base-RetinaNet.yaml with yaml.unsafe_load. Your machine may be at risk if the file contains malicious content.


...

[07/16 13:30:52 d2.checkpoint.detection_checkpoint]: [DetectionCheckpointer] Loading from /kaggle/working/dep229/model_final.pth ...


Loading config /usr/local/lib/python3.12/dist-packages/detectron2/model_zoo/configs/COCO-Detection/../Base-RetinaNet.yaml with yaml.unsafe_load. Your machine may be at risk if the file contains malicious content.


[07/16 13:30:53 d2.checkpoint.detection_checkpoint]: [DetectionCheckpointer] Loading from /kaggle/working/dep229/model_final.pth ...


dep MS test: 100%|██████████| 2000/2000 [15:03<00:00,  2.21it/s]

candidate pool @0.05: single-scale=8299 (4.149/img) | multi-scale=182715 (91.358/img) | THEM 174416 box


## 8. Rescore 229 (NGUYEN) tren 2 nguon: single (=229 anchor) vs multi-scale. Chan doan density.

In [9]:
R={"MIN_KEEP":0.20,"P_HI":0.55,"P_LO":0.25,"EPS":0.01}
W_DIFF,W_GEO=0.90,0.10

def rescore_229(cand, dep):
    """rescore 229 exact: p=0.9*s_diff+0.1*s_geo, demote p>=0.55, MIN_KEEP filter, overlap-suppress."""
    outp={}
    for stem,(cb,cs) in cand.items():
        if len(cb)==0: outp[stem]=(cb,cs); continue
        db,ds=dep.get(stem,(np.zeros((0,4),np.float32),np.zeros(0,np.float32)))
        sd=s_diff_single(cb,cs,db,ds); sg=geometry_scores(cb)
        p=np.clip(W_DIFF*sd+W_GEO*sg,0,1)
        new=np.zeros(len(cs),np.float32)
        for i,(s,pp) in enumerate(zip(cs,p)):
            if s<R["MIN_KEEP"]: new[i]=0.0
            elif pp>=R["P_HI"]: new[i]=R["EPS"]
            elif pp<=R["P_LO"]: new[i]=float(s)
            else:
                frac=(pp-R["P_LO"])/max(R["P_HI"]-R["P_LO"],1e-6); new[i]=max(R["EPS"], float(s)*(1-frac))
        keep=new>0.0
        eps_ids=np.where(new<=R["EPS"]+1e-6)[0]; strong=np.where(new>0.20)[0]
        if len(eps_ids) and len(strong):
            ov=iou_matrix(cb[eps_ids],cb[strong]).max(1); keep[eps_ids[ov>=0.20]]=False
        outp[stem]=(cb[keep], new[keep])
    return outp

pred_single = rescore_229(TEST_CAND,    TEST_DEP)      # = 229 anchor (single scale)
pred_ms     = rescore_229(TEST_CAND_MS, TEST_DEP_MS)   # multi-scale recall recovery

def dens(pred): 
    t=sum(len(s) for _,s in pred.values()); return t, t/len(test_files)
ts,ds_ = dens(pred_single); tm,dm = dens(pred_ms)
print(f"sau rescore 229: single={ts} ({ds_:.3f}/img)  <- =229 anchor")
print(f"                 multi-scale={tm} ({dm:.3f}/img)  THEM {tm-ts} box song qua rescore")
print("Doc: box THEM song qua rescore 229 = streak that thu hoi (rescore da loc poison).")
print("Neu multi-scale THEM box hop ly (khong no) -> nop MS. LB la trong tai (KHONG dung poison_mass).")


sau rescore 229: single=2582 (1.291/img)  <- =229 anchor
                 multi-scale=3404 (1.702/img)  THEM 822 box song qua rescore
Doc: box THEM song qua rescore 229 = streak that thu hoi (rescore da loc poison).
Neu multi-scale THEM box hop ly (khong no) -> nop MS. LB la trong tai (KHONG dung poison_mass).


## 9. Ghi 1 file submission.csv (multi-scale) + anchor 229 single-scale

In [10]:
sample=pd.read_csv(SAMPLE_SUB, dtype={"image_id":str})
def fmt(b,s):
    parts=[]
    for (x1,y1,x2,y2),sc in zip(b,s):
        x1,y1=float(np.clip(x1,0,IMG_W)),float(np.clip(y1,0,IMG_H)); x2,y2=float(np.clip(x2,0,IMG_W)),float(np.clip(y2,0,IMG_H))
        w,h=x2-x1,y2-y1
        if w>0 and h>0 and sc>0: parts+=[f"{float(sc):.6f}",f"{x1:.2f}",f"{y1:.2f}",f"{w:.2f}",f"{h:.2f}"]
    return " ".join(parts) if parts else " "
def write_sub(pred, path):
    sub=sample.copy(); sub["prediction_string"]=sub["image_id"].map(lambda i: fmt(*pred.get(str(i),(np.zeros((0,4)),np.zeros(0)))))
    sub["prediction_string"]=sub["prediction_string"].fillna(" ")
    sub.loc[sub["prediction_string"].str.strip()=="","prediction_string"]=" "
    sub.to_csv(path,index=False); return sum(len(s) for _,s in pred.values())

write_sub(pred_single, str(WORK/"submission_anchor229.csv"))   # A/B anchor
tot=write_sub(pred_ms, str(WORK/"submission.csv"))             # DEFAULT = multi-scale recall
print(f"saved submission.csv (multi-scale) | dets={tot} ({tot/len(test_files):.3f}/img)")
print("saved submission_anchor229.csv (single-scale = 229). Nop submission.csv, so LB voi anchor229.")
print("Neu MS < 229 -> recall recovery la huong dung (cat FN). Neu >= 229 -> box them la FP/poison, chot 229.")


saved submission.csv (multi-scale) | dets=3404 (1.702/img)
saved submission_anchor229.csv (single-scale = 229). Nop submission.csv, so LB voi anchor229.
Neu MS < 229 -> recall recovery la huong dung (cat FN). Neu >= 229 -> box them la FP/poison, chot 229.
